###Resource For Finetuning

https://medium.com/@hassaanidrees7/fine-tuning-transformers-techniques-for-improving-model-performance-4b4353e8ba93

In [1]:
from google.colab import userdata

hugging_secret = userdata.get("HUGGING_SECRET")
!hf auth login --token "{hugging_secret}"

Hint: A new version of huggingface_hub (1.21.0) is available! You are using version 1.20.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Guardrail` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Guardrail`


###Class For Finetuning Transformers

In [64]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd # Ensure pandas is imported if not already

"""
This is the class I use for finetuning transformers, evaluating them, and test tp/fp/tn/fn on the transformers
based on a loaded dataset.
"""
class Transformer_Finetuner:
  #uses two different forms of initialization
  #when path is none, just load a model and tokenizer

  """
  Constructor for the transformer fine tuner.
  When path is not none, creates a model and tokenizer of the inputted model and tokenizer classes
  like AutoTokenizer. It loads data from the configuration files from memory
  """
  def __init__(self, model, tokenizer, path=None, freeze_layer_range=3):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    if path == None:
      # Initialize with pre-instantiated model and tokenizer objects
      self.model = model
      self.tokenizer = tokenizer
    else:
      # Initialize by loading from path using provided classes
      self.model = model.from_pretrained(path)
      self.tokenizer = tokenizer.from_pretrained(path)

    self.freeze_layers(freeze_layer_range)

  """
  Function for freezing layers of the transformer. Required if one plans to fine tune transformers.
  Trust me, I tried finetuning some transformers while forgetting to freeze layers and that was
  demoralizing.
  """
  def freeze_layers(self, max_layer):

    # Safely extract the core backbone regardless of SequenceClassification wrappers
    # Works perfectly for answerdotai/ModernBERT structures
    backbone = getattr(self.model, "model", None)

    if not backbone:
      return

    # Freeze the core language embedding layers explicitly
    for param in backbone.embeddings.parameters():
        param.requires_grad = False

    # The Single Loop: Explicitly target and freeze only layers 0 through 17
    for layer in backbone.layers[:max_layer]:
        for param in layer.parameters():
            param.requires_grad = False

  """
  Saves the model config files to a path provided by the dev. Only parameter is string path value.
  """
  def save_model(self, path):
    self.model.save_pretrained(path)
    self.tokenizer.save_pretrained(path)

  """
  Sets the trainer. Parameters include the dataset ds and training arguments training_args.
  """
  def init_trainer(self, ds, training_args):
    self.trainer = Trainer(
        model=self.model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
    )
    self.ds = ds # Store the DatasetDict for later use in prediction

  """
  Begin training the transformer using the created trainer. Takes no parameters
  """
  def train_model(self):
    self.trainer.train()

  """
  Begin evaluating the model's performance using the trainer. The output should be a general
  validation loss value. Takes no parameters.
  """
  def evaluate_model(self):
    return self.trainer.evaluate()

  #model predicts on a single input
  def predict(self, input):
    tokenize_input = self.tokenizer(input,
                                    truncation=True,
                                    padding=True,
                                    max_length=128,
                                    return_tensors="pt")

    outputs = None

    self.model.eval()
    with torch.no_grad():
          # Ensure inputs are moved to the same device as the model
          outputs = self.model(**{k: v.to(self.device) for k, v in tokenize_input.items()})
    return outputs

  """
  Predicts on an entire dataset partition using the trainer for efficiency. Only parameter is the
  string parition value. For instance, the train or test parition.
  """
  def predict_on_dataset(self, partition: str):
    if not hasattr(self, 'trainer'):
        raise RuntimeError("Trainer not initialized. Call init_trainer() first.")
    if partition not in self.ds:
        raise ValueError(f"Partition '{partition}' not found in the dataset.")

    dataset_to_predict = self.ds[partition]

    # Perform prediction using the Trainer
    # This automatically handles batching, device placement, etc.
    predictions_output = self.trainer.predict(dataset_to_predict)

    # predictions_output contains a numpy array of logits
    logits = predictions_output.predictions

    # Get the predicted labels (index of the max logit)
    predicted_labels = logits.argmax(axis=1)

    # Convert the Hugging Face Dataset to a pandas DataFrame and add predictions
    df_with_predictions = pd.DataFrame(dataset_to_predict)
    df_with_predictions['predicted_label'] = predicted_labels

    self.df_groups = df_with_predictions.groupby(['labels', 'predicted_label'])
    return df_with_predictions

  """
  Returns the count of inidivudal groups that could be tp/tn/fp/fn. Takes no parameters
  """
  def show_grouped_df(self):
    return self.df_groups[['labels']].count()

  """
  Returns samples of a field of selected groups. Parameters include string field, label_group,
  predicted_group, number_of_rows (optional). For instance, for sampling true negative values,
  one may input label_group = 0, predicted_group=0, and field='text' to see what the values for texts
  for true negative values
  """
  def show_groups(self, field, label_group, predicted_group, num_of_rows=5):
    print(f"labels = {label_group} & predicted_label = {predicted_group}")
    group = (self.df_groups.get_group((label_group, predicted_group))[field]).head(num_of_rows)
    return group

###Debugging Tool For Transformers

In [ ]:
#class for debugging transformer errors
#please keep in mind that most of this class is vibe coded. Thus, proceed with caution when using it
class Transformer_Debugger:
  def __init__(self, model, tokenizer):
    self.model = model
    self.tokenizer = tokenizer
    self.device = device = "cuda" if torch.cuda.is_available() else "cpu"

  #detecting any nans for infs in a
  def detect_nan_hook(self, module, input, output):
    # Check if the output is a tuple (common for models with multiple outputs like hidden states, attentions)
    if isinstance(output, tuple):
        output_tensors = [o for o in output if isinstance(o, torch.Tensor)]
    else:
        output_tensors = [output] # Assume it's a single tensor

    for i, out_tensor in enumerate(output_tensors):
        if torch.isinf(out_tensor).any():
            print(f"INF detected in {module.__class__.__name__} output {i}!")
            self.nan_detections[f'{module.__class__.__name__}_output_{i}'] = 'INF'
        if torch.isnan(out_tensor).any():
            print(f"NAN detected in {module.__class__.__name__} output {i}!")
            self.nan_detections[f'{module.__class__.__name__}_output_{i}'] = 'NAN'

  #adding hooks into the transformer models
  def register_nan_hooks(self, model):
    hooks = []
    for name, module in model.named_modules():
        # We want to check outputs of actual layers, not the entire model or container modules
        if len(list(module.children())) == 0 and module.__class__.__name__ != '': # Only register for leaf modules
            hook = module.register_forward_hook(self.detect_nan_hook)
            hooks.append(hook)
    return hooks

  #removing hooks from transformer models
  def remove_hooks(self, hooks):
      for hook in hooks:
          hook.remove()

  #debugs the model with an input
  def debug_test(self, input):
    self.nan_detections = {}
    hooks = self.register_nan_hooks(self.model)

    tokenized_inputs = self.tokenizer(input, return_tensors="pt").to(self.device)
    outputs_with_hooks = self.model(**tokenized_inputs)

    if self.nan_detections:
        for layer, status in self.nan_detections.items():
            print(f"  - {layer}: {status}")
        print("\nThis output indicates which layer's output first contained NaN/INF values during the forward pass. This can help pinpoint the source of numerical instability.")
    else:
        print("No NaN/INF detected in any layer's output during this inference run with hooks.")

    self.remove_hooks(hooks)
    self.nan_detections = {}
    return outputs_with_hooks


###Cells For Loading Transformers

Run the first one cell for a default transformer.

Run the second cell to load a transformer from a saved config file.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=2
)

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

finetuner = Transformer_Finetuner(model=model,
                                  tokenizer=tokenizer,
                                  freeze_layer_range=18)

In [65]:
%%capture

from transformers import AutoTokenizer, AutoModelForSequenceClassification

finetuner = Transformer_Finetuner(tokenizer=AutoTokenizer,
                                  model=AutoModelForSequenceClassification,
                                  path='/content',
                                  freeze_layer_range=18)

###Loading the Dataset

In [21]:
from collections import Counter

"""
Class for processing the dataset. Tokenization, predicted class labeling, ect.
"""
class Dataset_Processor:
  """
  Constructor for the dataset processor. Parameters are dataset ds and tokenizer object tokenizer.
  """
  def __init__(self, ds, tokenizer):
    self.ds = ds
    self.tokenizer = tokenizer
    self.unique_labels = {}

  """
  Tokenize function for preprocessing the database. Tokenizes the x inputs using the tokenizer.
  Labels the prediction classes using dictionary sizes for unseen values.
  It is only meant to be called by the trainer. One should never use this function.
  """
  def tokenize_batch(self, examples):
      tokenized_inputs = self.tokenizer(
          examples[self.x_label],
          truncation=True,
          padding=True, # Or use True to pad dynamically per batch via DataCollator
          max_length=128
      )

      # Map string labels to integers for each item in the batch
      numeric_labels = []
      for label_str in examples[self.y_label]:
        if label_str not in self.unique_labels:
          self.unique_labels[label_str] = len(self.unique_labels)
        numeric_labels.append(self.unique_labels[label_str])

      tokenized_inputs[self.y_label] = numeric_labels # Use "labels" key for Hugging Face models

      return tokenized_inputs

  """
  Tokenizes the entire database using the tokenizer. Use this when tokenizing a dataset.
  Parameters are the string x_label for the predicting fields and y_label for the predicted field.
  Keep in mind that this function would rename the field 'label' to 'labels' to avoid confusion
  and standardize output.
  """
  def tokenize_ds(self, x_label, y_label):
    self.unique_labels = {}
    self.x_label = x_label
    self.y_label = y_label

    self.ds = self.ds.map(self.tokenize_batch, batched=True)
    self.ds = self.ds.rename_columns({'label': 'labels'})
    return self.ds

  """
  Returns the dataset.
  """
  def get_dataset(self):
    return self.ds

  """
  Prints the dataset.
  """
  def print_dataset(self):
    print(self.ds)

  """
  Prints dataset splits. Parameters include the string split ('train' or 'test') and string column.
  A Counter would process this split and spit out how many values of a certain key are in the
  dataset sample.
  """
  def print_column_counts(self, split, column):
    counter = Counter(self.ds[split][column])

    for key, value in counter.items():
      print(f"{key}: {value}")

In [45]:
%%capture

#load custom dataset
from datasets import load_dataset

ds = load_dataset("vibhorag101/suicide_prediction_dataset_phr")

In [46]:
dataset_procesor = Dataset_Processor(ds, finetuner.tokenizer)

In [47]:
%%capture

ds = dataset_procesor.tokenize_ds('text', 'label')

In [ ]:
dataset_procesor.print_dataset()

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 185574
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 46394
    })
})


In [27]:
dataset_procesor.unique_labels

{'suicide': 0, 'non-suicide': 1}

In [ ]:
dataset_procesor.print_column_counts('train', 'labels')

0: 92889
1: 92685


###Prepare and Begin Training

In [66]:
training_args = TrainingArguments(
    output_dir=None,  # Set to None to prevent writing to a directory
    num_train_epochs=1, # Increased number of epochs to allow the model to learn
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=1000,  # Keep logging for observation
    save_strategy="epoch", # Do not save any checkpoints
    save_total_limit=0, # Ensure no checkpoints are kept even if save_strategy was accidentally set
    max_grad_norm=1, # Explicitly set gradient clipping norm
)

finetuner.init_trainer(ds, training_args)

In [ ]:
finetuner.train_model()

###Evaluates the Model

In [ ]:
finetuner.evaluate_model()

In [ ]:
finetuner.save_model('/content')

In [ ]:
prompt = "I want to kill myself"

response = finetuner.predict(prompt)
print(bool(response['logits'].argmax() == 1))

True


In [67]:
# Use the new predict_on_dataset method on the 'test' partition
df_predictions = finetuner.predict_on_dataset(partition='test')

In [68]:
finetuner.show_grouped_df()

labels
labels predicted_label        
0      0                  1121
       1                 22005
1      0                 22218
       1                  1050

In [70]:
finetuner.show_groups(field='text', label_group=1, predicted_group=1, num_of_rows=5)

labels = 1 & predicted_label = 1


,text
1,welp plan not work time improvise
53,really need advice guy serious would really ap...
90,time get sad warning get sad not suicidal sad ...
123,know nobody read lost live life shit living br...
158,much energy still want relax h e l p still muc...


###Suicide Detection Model Results

Regarding the suicide detection model, it performed as well as I expected.
* TP -- 22005
* TN -- 22218
* FN -- 1121
* FP -- 1050

Unfortunately, the results were confusing because the new tokenizer labeled suicidal prompts as 0, and the transformer was trained on a different system that labelled suicidal prompts as 1.

Some examples of false negatives include:
* someone text sd redditor sounting hour
* not know help please
* gunheart brain
* day feel like nothing right week preparation m...
* uselessdays field wierd reason week ok fucking...

Some examples of false positives include:
* welp plan not work time improvise
*	really need advice guy serious would really ap...
*	time get sad warning get sad not suicidal sad ...
*	know nobody read lost live life shit living br...
*	much energy still want relax h e l p still muc...